In [1]:
import pandas as pd
import numpy as np

In [7]:
import os
print(os.getcwd())
print(os.path.exists(r"C:/Users/moham/MTech/FedEx---Digital-and-Sustainable-Supply-Chain-Modelling-and-Analytics/bronze_kpi_weekly_final.csv"))



/content
False


In [13]:
# Weekly file
weekly = pd.read_excel('Weekly 202201-202412(1).xlsx')

# Aug actuals file
aug = pd.read_excel('Aug actuals with DOM (1).xlsx')

FileNotFoundError: [Errno 2] No such file or directory: 'Weekly 202201-202412(1).xlsx'

In [3]:
weekly.shape, aug.shape

((168897, 15), (271802, 25))

# Lane Harmonization

In [4]:
# Lane mapping dictionary
lane_map = {
    "EU": "Europe",
    "AS": "APAC",
    "ME": "MEISA",
    "NA": "Americas",
    "LA": "Americas"}

In [5]:
# apply mapping to weekly file
weekly['Dest_Lane'] = weekly['Dest_Lane'].replace(lane_map)

In [6]:
# apply lane recovery rule to aug file
aug.loc[(aug['Lane_'].isna()) & (aug['Lane']=='Americas'), 'Lane_'] = 'LA'

In [7]:
# Harmonize Lane_
aug['Dest_Lane'] = aug['Lane_'].replace(lane_map)

# Unit Standardization

In [8]:
# weekly files
weekly['aPounds'] = pd.to_numeric(weekly['aPounds'], errors='coerce')
weekly['Pounds'] = pd.to_numeric(weekly['Pounds'], errors='coerce')
weekly['PACKS'] = pd.to_numeric(weekly['PACKS'], errors='coerce')
weekly['Shipments'] = pd.to_numeric(weekly['Shipments'], errors='coerce')

In [9]:
# aug files
aug = aug.rename(columns={'Packs': 'PACKS', 'aLbs': 'aPounds'})

In [10]:
# convert to numeric
aug['aPounds'] = pd.to_numeric(aug['aPounds'], errors='coerce')
aug['PACKS'] = pd.to_numeric(aug['PACKS'], errors='coerce')
aug['Shipments'] = pd.to_numeric(aug['Shipments'], errors='coerce')

In [11]:
# adding kg column
weekly['aKg'] = weekly['aPounds'] * 0.453592
aug['aKg'] = aug['aPounds'] * 0.453592

# Creating Canonical Bronze Table

In [12]:
weekly['ship_date'] = pd.NaT # weekly aggregated

weekly_bronze = weekly[[
    "ship_date",
    "FY",
    "WeekNbr",
    "Orig_Ctry",
    "ORIG_RAMP",
    "Business_Region",
    "Dest_Lane",
    "Product_Code",
    "PACKS",
    "Shipments",
    "aPounds"]]

In [14]:
# Transform aug file
aug["ship_date"] = pd.to_datetime(aug["ShipDate"], errors="coerce")
aug["Year"] = aug["yyyymm"].astype(str).str[:4]
aug["FY"] = "FY" + aug["Year"].str[-2:]
aug["Product_Code"] = aug["Product"]

In [16]:
aug_bronze = aug[[
    "ship_date",
    "FY",
    "WeekNbr",
    "Orig_Ctry",
    "ORIG_RAMP",
    "Business_Region",
    "Dest_Lane",
    "Product_Code",
    "PACKS",
    "Shipments",
    "aPounds"]]

# Combining files

In [17]:
bronze_kpi = pd.concat([weekly_bronze, aug_bronze], ignore_index=True)

In [18]:
# fill null measures
bronze_kpi[["PACKS", "Shipments", "aPounds"]] = bronze_kpi[["PACKS", "Shipments", "aPounds"]].fillna(0)

In [19]:
bronze_kpi = bronze_kpi[[
    "ship_date",
    "FY",
    "WeekNbr",
    "Orig_Ctry",
    "ORIG_RAMP",
    "Business_Region",
    "Dest_Lane",
    "Product_Code",
    "PACKS",
    "Shipments",
    "aPounds"]]

# Time coverage + Continuity grid

In [20]:
# Unique Keys
keys = bronze_kpi[["ORIG_RAMP", "Dest_Lane", "Product_Code"]].drop_duplicates()

In [21]:
# Unique time combination
time_dim = bronze_kpi[["FY", "WeekNbr"]].drop_duplicates()

In [22]:
# Cross join
keys["key"] = 1
time_dim["key"] = 1

full_grid = keys.merge(time_dim, on="key").drop("key", axis=1)

In [23]:
# Merge with Bronze
bronze_full = full_grid.merge(bronze_kpi, on=["ORIG_RAMP", "Dest_Lane", "Product_Code", "FY", "WeekNbr"], how="left")

In [24]:
# Filling missing values with zero
bronze_full[["PACKS", "Shipments", "aPounds"]] = bronze_full[["PACKS", "Shipments", "aPounds"]].fillna(0)

In [25]:
# Saving Final dataset
bronze_full.to_csv("bronze_kpi_weekly_final.csv", index=False)

In [29]:
bronze_full.sample(10)

,ORIG_RAMP,Dest_Lane,Product_Code,FY,WeekNbr,ship_date,Orig_Ctry,Business_Region,PACKS,Shipments,aPounds
350912,BLR,Americas,IP,FY24,33,2024-08-13,IN,IN,1.0,1.0,1.10200
335247,BLR,APAC,IP,FY24,33,2024-08-13,IN,IN,1.0,1.0,2.64480
84681,DXB,MEISA,IP,FY25,26,NaT,UZ,MEA,7.0,7.0,4.62966
287487,BOM,APAC,IP,FY24,33,2024-08-13,IN,IN,1.0,1.0,7.93440
388727,DEL,Americas,IP,FY24,35,2024-08-31,IN,IN,1.0,1.0,46.60000
396478,DEL,Americas,IP,FY24,34,2024-08-22,IN,IN,2.0,2.0,5.73040
227480,DXB,APAC,IP,FY24,46,NaT,NG,MEA,3.0,3.0,21.60508
381365,BOM,Europe,IP,FY24,33,2024-08-12,IN,IN,1.0,1.0,6.61200
5930,DXB,Americas,IP,FY24,35,2024-08-27,LK,MEA,1.0,1.0,37.10000
305830,BOM,MEISA,IP,FY24,35,2024-08-28,IN,IN,12.0,1.0,446.97120


In [27]:
bronze_full.shape

(447006, 11)